In [ ]:

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

In [ ]:
import os


load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [ ]:
def display(text):
    try:
        Console().print(text)
    except Exception as e:
        print(text)   

In [ ]:
base_url = "https://openrouter.ai/api/v1"

In [ ]:
openai = OpenAI(base_url=base_url, api_key=OPENROUTER_API_KEY)

In [ ]:
tasks = []
completed = []

In [ ]:
def get_checklist_report() -> str:
    result = ""
    for index, task in enumerate(tasks):
        if completed[index]:
            result += f"Task #{index + 1}: [green][strike]{task}[/strike][/green]\n"
        else:
            result += f"Task #{index + 1}: {task}\n" 
    display(result)
    return result

In [ ]:
get_checklist_report()

In [ ]:
def create_tasks(descs: list[str]) -> None:
    tasks.extend(descs)
    completed.extend([False] * len(descs))
    get_checklist_report()

In [ ]:
def mark_completed(task_index: int, completion_notes: str) -> str:
    if 1 <= task_index <= len(tasks):
        completed[task_index - 1] = True
    else:
       return f"Invalid task index: {task_index}. Please provide a valid task number."

    Console().print(completion_notes)
    return get_checklist_report()

In [ ]:
tasks, completed = [], []

create_tasks(["Buy groceries", "Finish week 1", "Eat banana"])

In [ ]:
mark_completed(2, "Finished week 1 successfully!")

In [ ]:
create_tasks_json = {
    "name": "create_tasks",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type" : "object",
        "properties" : {
            "descs": {
                "type": "array",
                "items": {"type": "string"},
                "description": "A list of task descriptions to be added to the checklist",
            },    
        },
        "required": ["descs"],
        "additionalProperties": False,
    }
}

In [ ]:
mark_completed_json = {
    "name": "mark_completed",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        "type" : "object",
        "properties" : {
            "task_index": {
                "type": "integer",
                "description": "The index of the task to be marked as completed (1-based index)",
            },
            "completion_notes": {
                "type": "string",
                "description": "Notes or comments about the completion of the task",
            }, 
        },
        "required": ["task_index", "completion_notes"],
        "additionalProperties": False,
    },
}

In [ ]:
tools = [{
    "type": "function",
    "function": create_tasks_json
},{
    "type": "function",
    "function": mark_completed_json
}]

In [ ]:
def handle_tool_call(tool_calls):
    results = []
    print(tool_calls)
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if tool_name == "create_tasks":
            result = create_tasks(**arguments)
        elif tool_name == "mark_completed":
            result = mark_completed(**arguments)
        else:
            result = {}

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
def loop(messages):
    response = openai.chat.completions.create(
        model="openai/gpt-5.4-mini",
        messages=messages,
        tools=tools,     max_tokens=2000,
)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        print(f"Tool call: {message.tool_calls[0].function.name} with arguments {message.tool_calls[0].function.arguments}")
        tool_calls = message.tool_calls
        result = handle_tool_call(tool_calls)
        messages.append(message)
        messages.extend(result)
        response = openai.chat.completions.create(
            model="openai/gpt-5.4-mini",
            messages=messages,
            tools=tools,     
            max_tokens=2000,
)
    display(response.choices[0].message.content)

In [ ]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
tasks, completed = [], []
loop(messages)